# Explicação das funções implementadas para BER na saída do equalizador

Este notebook contém apenas a explicação teórica e a análise das funções implementadas para calcular a BER analítica na saída de equalizadores FFE e DFE.

O bloco implementado corresponde à etapa final do modelo analítico:

$$\overline{SNR}(f) \longrightarrow SNR_{FFE},\; SNR_{DFE} \longrightarrow BER$$

Isto significa que o notebook assume que a SNR espectral dobrada, $\overline{SNR}(f)$, já foi calculada anteriormente.


## 1. Contexto do modelo

No modelo analítico do artigo, primeiro calcula-se uma SNR dependente da frequência. Após considerar o efeito de dobramento espectral causado pela amostragem no receptor, obtém-se:

$$\overline{SNR}(f)$$

Essa é a SNR espectral efetiva vista pelo equalizador dentro da banda:

$$-\frac{1}{2T} \leq f \leq \frac{1}{2T}$$

onde $T$ é o período de símbolo.

A partir de $\overline{SNR}(f)$, calculam-se duas SNRs efetivas: uma para o FFE e outra para o DFE. Depois, essas SNRs são convertidas em BER por meio da fórmula aproximada para modulação M-PAM em ruído gaussiano.


## 2. Bibliotecas necessárias

As funções usam apenas:

- `numpy`, para operações vetoriais, logaritmo, raiz e integração numérica;
- `scipy.special.erfc`, para a função complementar do erro.

Não há simulação temporal nesta parte. Não se usa `ffe()` nem `dfe()` aplicados ao sinal no tempo.


In [ ]:
import numpy as np
from scipy.special import erfc


## 3. Função `ber_from_snr_m_pam`

A função implementada foi:

```python
def ber_from_snr_m_pam(snr_linear, M):
    snr_linear = np.asarray(snr_linear)

    ber = ((M - 1) / (M * np.log2(M))) * erfc(
        np.sqrt((3 * snr_linear) / (2 * (M**2 - 1)))
    )

    return ber
```

Ela calcula a BER aproximada para uma modulação M-PAM a partir da SNR em escala linear.


### 3.1 Fórmula implementada

A fórmula usada é:

$$BER \approx \frac{M-1}{M\log_2(M)}\operatorname{erfc}\left(\sqrt{\frac{3\,SNR}{2(M^2-1)}}\right)$$

onde:

- $M$ é a ordem da modulação PAM;
- $SNR$ é a SNR efetiva na saída do equalizador, em escala linear;
- $\operatorname{erfc}(\cdot)$ é a função complementar do erro.

Para 4-PAM, deve-se usar:

$$M=4$$

A SNR não deve estar em dB. Se a SNR estiver em dB, a conversão correta é:

$$SNR_{linear}=10^{SNR_{dB}/10}$$


### 3.2 Análise da implementação

A linha:

```python
snr_linear = np.asarray(snr_linear)
```

transforma a entrada em array NumPy. Isso permite que a função aceite tanto um único valor de SNR quanto um vetor de valores.

O trecho:

```python
((M - 1) / (M * np.log2(M)))
```

implementa o fator:

$$\frac{M-1}{M\log_2(M)}$$

Esse fator ajusta a expressão para uma constelação M-PAM.

O trecho:

```python
np.sqrt((3 * snr_linear) / (2 * (M**2 - 1)))
```

implementa o argumento da função complementar do erro:

$$\sqrt{\frac{3\,SNR}{2(M^2-1)}}$$

Como esse termo cresce com a SNR, a BER diminui quando a SNR aumenta.


In [ ]:
def ber_from_snr_m_pam(snr_linear, M):
    """
    Calcula a BER aproximada para M-PAM a partir da SNR em escala linear.

    Fórmula:
        BER ≈ (M - 1)/(M log2(M)) * erfc(
            sqrt(3*SNR / (2*(M^2 - 1)))
        )

    Parâmetros
    ----------
    snr_linear : float ou np.ndarray
        SNR em escala linear, não em dB.

    M : int
        Ordem da modulação PAM.

    Retorna
    -------
    ber : float ou np.ndarray
        BER estimada.
    """
    snr_linear = np.asarray(snr_linear)

    ber = ((M - 1) / (M * np.log2(M))) * erfc(
        np.sqrt((3 * snr_linear) / (2 * (M**2 - 1)))
    )

    return ber


## 4. Função `snr_equalizer_output_ffe`

A função implementada foi:

```python
def snr_equalizer_output_ffe(snr_folded, freqs_base, T):
    integrand = 1 / (snr_folded + 1)
    integral = np.trapz(integrand, freqs_base)

    return (1 / T) * (1 / integral) - 1
```

Ela calcula a SNR efetiva na saída de um equalizador FFE a partir da SNR espectral dobrada, $\overline{SNR}(f)$.


### 4.1 Fórmula implementada

A expressão teórica do FFE é:

$$SNR_{FFE}=\frac{1}{T}\left[\int_{-\frac{1}{2T}}^{\frac{1}{2T}}\frac{1}{\overline{SNR}(f)+1}\,df\right]^{-1}-1$$

Os parâmetros da função têm a seguinte interpretação:

- `snr_folded`: vetor com os valores de $\overline{SNR}(f)$;
- `freqs_base`: vetor de frequências usado na integração;
- `T`: período de símbolo.

O vetor `freqs_base` deve cobrir a banda:

$$-\frac{1}{2T} \leq f \leq \frac{1}{2T}$$

ou, equivalentemente:

$$-\frac{R_s}{2} \leq f \leq \frac{R_s}{2}$$

com:

$$R_s=\frac{1}{T}$$


### 4.2 Análise da implementação

A linha:

```python
integrand = 1 / (snr_folded + 1)
```

implementa o integrando:

$$\frac{1}{\overline{SNR}(f)+1}$$

Esse formato penaliza fortemente regiões onde a SNR é baixa. Essa característica é compatível com o FFE, pois equalizadores feed-forward podem amplificar o ruído quando tentam compensar frequências muito atenuadas do canal.

A linha:

```python
integral = np.trapz(integrand, freqs_base)
```

calcula numericamente a integral em frequência usando a regra dos trapézios. Matematicamente, ela aproxima:

$$\int_{-\frac{1}{2T}}^{\frac{1}{2T}}\frac{1}{\overline{SNR}(f)+1}\,df$$

A linha final:

```python
return (1 / T) * (1 / integral) - 1
```

implementa:

$$\frac{1}{T}\left[\int \frac{1}{\overline{SNR}(f)+1}\,df\right]^{-1}-1$$

Portanto, a função está consistente com a expressão analítica para o FFE.


In [ ]:
def snr_equalizer_output_ffe(snr_folded, freqs_base, T):
    integrand = 1 / (snr_folded + 1)
    integral = np.trapz(integrand, freqs_base)

    return (1 / T) * (1 / integral) - 1


## 5. Função `snr_equalizer_output_dfe`

A função implementada foi:

```python
def snr_equalizer_output_dfe(snr_folded, freqs_base, T):
    integrand = np.log(snr_folded + 1)
    integral = np.trapz(integrand, freqs_base)

    return np.exp(T * integral) - 1
```

Ela calcula a SNR efetiva na saída de um equalizador DFE a partir da mesma SNR espectral dobrada, $\overline{SNR}(f)$.


### 5.1 Fórmula implementada

A expressão teórica do DFE é:

$$SNR_{DFE}=\exp\left[T\int_{-\frac{1}{2T}}^{\frac{1}{2T}}\log\left(\overline{SNR}(f)+1\right)\,df\right]-1$$

Os parâmetros são os mesmos da função do FFE:

- `snr_folded`: vetor com $\overline{SNR}(f)$;
- `freqs_base`: eixo de frequências da integração;
- `T`: período de símbolo.

A diferença está na forma da média espectral. O DFE usa uma média logarítmica de $\overline{SNR}(f)+1$.


### 5.2 Análise da implementação

A linha:

```python
integrand = np.log(snr_folded + 1)
```

implementa:

$$\log\left(\overline{SNR}(f)+1\right)$$

A linha:

```python
integral = np.trapz(integrand, freqs_base)
```

aproxima numericamente:

$$\int_{-\frac{1}{2T}}^{\frac{1}{2T}}\log\left(\overline{SNR}(f)+1\right)\,df$$

A linha final:

```python
return np.exp(T * integral) - 1
```

implementa:

$$\exp\left[T\int \log\left(\overline{SNR}(f)+1\right)\,df\right]-1$$

Portanto, a função está consistente com a expressão analítica para o DFE.

Como o DFE lida de forma diferente com interferência intersimbólica, ele tende a apresentar SNR efetiva maior ou igual à do FFE em canais fortemente seletivos, dentro das hipóteses do modelo analítico.


In [ ]:
def snr_equalizer_output_dfe(snr_folded, freqs_base, T):
    integrand = np.log(snr_folded + 1)
    integral = np.trapz(integrand, freqs_base)

    return np.exp(T * integral) - 1


## 6. Observações sobre unidades e escala

As três funções dependem de uma convenção importante: a SNR deve estar em escala linear.

Conversão de dB para linear:

$$SNR_{linear}=10^{SNR_{dB}/10}$$

Conversão de linear para dB:

$$SNR_{dB}=10\log_{10}(SNR_{linear})$$

A BER é adimensional.

O eixo `freqs_base` deve estar em Hz, e o período de símbolo `T` deve estar em segundos. Assim, o produto entre `T` e uma integral em frequência fica adimensional, como exigido pelas fórmulas.


## 7. Interpretação conceitual das três funções

As funções `snr_equalizer_output_ffe` e `snr_equalizer_output_dfe` não simulam equalizadores no tempo. Elas fazem uma estimativa analítica da SNR efetiva após o equalizador.

A função do FFE usa:

$$\frac{1}{\overline{SNR}(f)+1}$$

Por isso, regiões espectrais com SNR baixa têm grande impacto no resultado.

A função do DFE usa:

$$\log\left(\overline{SNR}(f)+1\right)$$

Isso produz um comportamento mais próximo de uma média geométrica espectral.

Depois que uma SNR efetiva é obtida, `ber_from_snr_m_pam` transforma essa SNR em uma BER estimada para M-PAM.


## 8. Fluxo correto de uso

Depois que o grupo tiver calculado `snr_folded`, `freqs_base` e `T`, o uso esperado é:

```python
snr_ffe = snr_equalizer_output_ffe(snr_folded, freqs_base, T)
snr_dfe = snr_equalizer_output_dfe(snr_folded, freqs_base, T)

ber_ffe = ber_from_snr_m_pam(snr_ffe, M=4)
ber_dfe = ber_from_snr_m_pam(snr_dfe, M=4)
```

A sequência conceitual é:

$$\overline{SNR}(f) \rightarrow SNR_{FFE} \rightarrow BER_{FFE}$$

$$\overline{SNR}(f) \rightarrow SNR_{DFE} \rightarrow BER_{DFE}$$

O parâmetro `M` deve ser escolhido de acordo com a modulação. Para o caso mais usado no artigo:

$$M=4$$


## 9. Conclusão

As três funções implementam a etapa final do modelo analítico:

$$\overline{SNR}(f) \rightarrow SNR_{FFE/DFE} \rightarrow BER$$

A função `snr_equalizer_output_ffe` implementa a expressão analítica da SNR na saída do FFE.

A função `snr_equalizer_output_dfe` implementa a expressão analítica da SNR na saída do DFE.

A função `ber_from_snr_m_pam` implementa a aproximação de BER para M-PAM a partir da SNR efetiva.

Portanto, se `snr_folded`, `freqs_base` e `T` estiverem corretos, essas funções estão organizadas de forma compatível com o bloco analítico final usado no artigo.
